# Fashion Coat vs Dress Classification

Binary image classification on a Fashion-MNIST subset: distinguish **coats** from **dresses** using classical machine learning.

High-dimensional pixel inputs (784 features) make this a natural testbed for **dimensionality reduction** (PCA, LLE) combined with **SVM**, **Gaussian Naive Bayes**, and **LDA**.

**Outcome:** An SVM with PCA reduction reached strong validation accuracy (~96%) and ~93% held-out test accuracy in prior runs of this pipeline.

| Item | Detail |
|------|--------|
| Task | Binary classification |
| Input | 28×28 grayscale images (784 pixels) |
| Classes | 0 = Coat, 1 = Dress |
| Models | SVM, GaussianNB, LDA |
| Reduction | PCA, LLE |
| Metric | Accuracy (primary), confusion matrices |


## Setup

Load libraries, apply a consistent visual style, and define project paths.


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import LocallyLinearEmbedding as LLE
from sklearn.model_selection import train_test_split, GridSearchCV, ParameterGrid, cross_val_score
from sklearn.metrics import accuracy_score
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA

# Repo root (…/machine-learning-portfolio) so we can import shared style
REPO_ROOT = Path("../../..").resolve()
# Fall back: walk upward until we find common/portfolio_style.py
if not (REPO_ROOT / "common" / "portfolio_style.py").exists():
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "common" / "portfolio_style.py").exists():
            REPO_ROOT = candidate
            break
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from common.portfolio_style import (
    PALETTE,
    CLASS_COLORS,
    apply_mpl_style,
    plot_confusion as _plot_confusion,
    line_accuracy_chart as _line_accuracy_chart,
)

apply_mpl_style()

PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data" / "raw"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
np.random.seed(SEED)

CLASS_NAMES = {0: "Coat", 1: "Dress"}


def plot_confusion(y_true, y_pred, title, save_as=None):
    path = FIGURES_DIR / save_as if save_as else None
    return _plot_confusion(
        y_true,
        y_pred,
        title,
        class_names=[CLASS_NAMES[0], CLASS_NAMES[1]],
        save_as=path,
    )


def line_accuracy_chart(x, y, title, xlabel="Components", save_as=None):
    path = FIGURES_DIR / save_as if save_as else None
    return _line_accuracy_chart(x, y, title, xlabel=xlabel, save_as=path)


print("Project root:", PROJECT_ROOT)
print("Data directory:", DATA_DIR)




Project root: /home/runner/work/machine-learning-portfolio/machine-learning-portfolio/projects/fashion-coat-dress-classification
Data directory: /home/runner/work/machine-learning-portfolio/machine-learning-portfolio/projects/fashion-coat-dress-classification/data/raw


## Data loading and split

We use a **60 / 20 / 20** train / validation / test split. Validation guides model selection; the test set is touched only once at the end to estimate generalization.


In [2]:
data = pd.read_csv(DATA_DIR / "train.csv")

y = data["label"]
X = data.drop(columns=["label"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.25, random_state=SEED, stratify=y_train
)

print(f"Train: {X_train.shape[0]} | Validation: {X_val.shape[0]} | Test: {X_test.shape[0]}")
print(f"Features: {X_train.shape[1]} (28x28 pixels)")

model_accuracies = {}


Train: 900 | Validation: 300 | Test: 300
Features: 784 (28x28 pixels)


## Exploratory analysis

The training labels are roughly balanced. Pixel intensities sit in the range 0–255, with a heavy mass at black background pixels — typical for Fashion-MNIST clothing silhouettes.


In [3]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))

counts = y_train.value_counts().sort_index()
axes[0].bar(
    [CLASS_NAMES[i] for i in counts.index],
    counts.values,
    color=CLASS_COLORS,
    width=0.55,
    edgecolor="white",
)
axes[0].set_title("Class balance (training set)")
axes[0].set_ylabel("Count")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 5, str(v), ha="center", color=PALETTE["slate"], fontsize=10)

pixels = X_train.to_numpy().ravel()
axes[1].hist(pixels, bins=64, range=(0, 255), color=PALETTE["med_blue"], edgecolor="white")
axes[1].set_title("Pixel intensity distribution")
axes[1].set_xlabel("Intensity (0-255)")
axes[1].set_ylabel("Frequency")

fig.tight_layout()
fig.savefig(FIGURES_DIR / "01_eda_overview.png", bbox_inches="tight")
plt.show()

print(f"Min pixel: {pixels.min()} | Max pixel: {pixels.max()}")
print(f"Mean intensity: {pixels.mean():.1f}")



Min pixel: 0 | Max pixel: 255
Mean intensity: 81.1


Sample images confirm the class semantics: **class 0** looks like coats/jackets, **class 1** like dresses.


In [4]:
fig, axes = plt.subplots(2, 6, figsize=(11, 4.2))
fig.suptitle("Sample images by class", fontsize=14, y=1.02)

for row, label in enumerate([0, 1]):
    idxs = np.where(y_train.to_numpy() == label)[0]
    chosen = np.random.choice(idxs, size=6, replace=False)
    for col, idx in enumerate(chosen):
        ax = axes[row, col]
        ax.imshow(X_train.iloc[idx].to_numpy().reshape(28, 28), cmap="gray_r")
        ax.set_xticks([])
        ax.set_yticks([])
        if col == 0:
            ax.set_ylabel(CLASS_NAMES[label], rotation=0, labelpad=28, va="center")

fig.tight_layout()
fig.savefig(FIGURES_DIR / "02_sample_images.png", bbox_inches="tight")
plt.show()


## Baseline models (no dimensionality reduction)

We standardize features for SVM and LDA. Naive Bayes is fit on raw pixel intensities.


In [5]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)


### Support Vector Machine

SVMs find a maximum-margin decision boundary and work well for mid-sized, moderately separable problems. We tune kernel type, the soft-margin penalty `C`, and RBF `gamma`.


In [6]:
svm_param_grid = [
    {"kernel": ["linear"], "C": [0.1, 1, 10, 15, 20]},
    {"kernel": ["rbf"], "C": [1, 10], "gamma": [0.001, 0.01, 0.1, 0.2, 0.5]},
]

svm_search = GridSearchCV(
    SVC(), svm_param_grid, cv=3, scoring="accuracy", n_jobs=-1, verbose=0
)
svm_search.fit(X_train_s, y_train)
svm_model = svm_search.best_estimator_

print("Best SVM params:", svm_search.best_params_)

for name, Xs, ys in [("train", X_train_s, y_train), ("validation", X_val_s, y_val)]:
    pred = svm_model.predict(Xs)
    acc = accuracy_score(ys, pred)
    print(f"Accuracy ({name}): {acc:.4f}")
    if name == "validation":
        model_accuracies["SVM"] = acc
        plot_confusion(ys, pred, "SVM — validation", save_as="03_cm_svm.png")


Best SVM params: {'C': 1, 'gamma': 0.001, 'kernel': 'rbf'}
Accuracy (train): 0.9833
Accuracy (validation): 0.9167


Training accuracy is very high relative to validation, so the full-dimensional SVM is likely **overfitting** the pixel space. The validation confusion matrix is fairly balanced across the two classes.


### Gaussian Naive Bayes

Naive Bayes is fast and simple, but it assumes feature independence — unrealistic for neighboring pixels. We still include it as a strong baseline for comparison.


In [7]:
nb_model = GaussianNB()
nb_model.fit(X_train, y_train)

for name, Xs, ys in [("train", X_train, y_train), ("validation", X_val, y_val)]:
    pred = nb_model.predict(Xs)
    acc = accuracy_score(ys, pred)
    print(f"Accuracy ({name}): {acc:.4f}")
    if name == "validation":
        model_accuracies["Naive Bayes"] = acc
        plot_confusion(ys, pred, "Naive Bayes — validation", save_as="04_cm_nb.png")


Accuracy (train): 0.8767
Accuracy (validation): 0.8367


Naive Bayes lags behind SVM and systematically confuses coats for dresses more often — consistent with its independence assumption ignoring spatial structure.


### Linear Discriminant Analysis

LDA seeks a linear projection that maximizes between-class separation under Gaussian assumptions. Shrinkage helps when the covariance estimate is unstable in high dimensions.


In [8]:
lda_param_grid = {"solver": ["svd", "lsqr", "eigen"], "shrinkage": [None, "auto"]}
valid_lda_params = [
    p for p in ParameterGrid(lda_param_grid)
    if not (p["solver"] == "svd" and p["shrinkage"] is not None)
]

best_score, best_lda_params = 0.0, None
for params in valid_lda_params:
    try:
        lda = LDA(**params)
        lda.fit(X_train_s, y_train)
        score = accuracy_score(y_val, lda.predict(X_val_s))
        if score > best_score:
            best_score, best_lda_params = score, params
    except Exception:
        continue

print("Best LDA params:", best_lda_params)
lda_model = LDA(**best_lda_params)
lda_model.fit(X_train_s, y_train)

for name, Xs, ys in [("train", X_train_s, y_train), ("validation", X_val_s, y_val)]:
    pred = lda_model.predict(Xs)
    acc = accuracy_score(ys, pred)
    print(f"Accuracy ({name}): {acc:.4f}")
    if name == "validation":
        model_accuracies["LDA"] = acc
        plot_confusion(ys, pred, "LDA — validation", save_as="05_cm_lda.png")


Best LDA params: {'shrinkage': 'auto', 'solver': 'lsqr'}


Accuracy (train): 0.9889
Accuracy (validation): 0.9067


LDA approaches SVM performance. Training accuracy is still high, so regularization via dimensionality reduction may help further.


## Synthetic samples from class-conditional Gaussians

Using class means and covariances estimated on the train+validation pool, we sample five synthetic images per class. These are not photorealistic, but silhouettes remain recognizable.


In [9]:
X_pool = np.vstack([X_train.to_numpy(), X_val.to_numpy()])
y_pool = np.concatenate([y_train.to_numpy(), y_val.to_numpy()])

class_params = {}
for label in np.unique(y_pool):
    class_data = X_pool[y_pool == label]
    mean = class_data.mean(axis=0)
    cov = np.cov(class_data.T) + 1e-5 * np.eye(class_data.shape[1])
    class_params[label] = (mean, cov)

fig, axes = plt.subplots(2, 5, figsize=(10, 4.2))
fig.suptitle("Synthetic samples (Gaussian draws around class means)", fontsize=13)

rng = np.random.default_rng(SEED)
for row, label in enumerate(sorted(class_params)):
    mean, cov = class_params[label]
    for col in range(5):
        sample = rng.multivariate_normal(mean, cov)
        sample = np.clip(sample, 0, 255).reshape(28, 28)
        axes[row, col].imshow(sample, cmap="gray_r")
        axes[row, col].set_xticks([])
        axes[row, col].set_yticks([])
        if col == 0:
            axes[row, col].set_ylabel(CLASS_NAMES[label], rotation=0, labelpad=28, va="center")

fig.tight_layout()
fig.savefig(FIGURES_DIR / "06_synthetic_samples.png", bbox_inches="tight")
plt.show()


The generated images are blurrier than real Fashion-MNIST samples and rarely produce a clean black background, but coat vs dress outlines are usually still legible.


## Dimensionality reduction with PCA

For each classifier we search over PCA dimensionality and model hyperparameters, scoring candidates on the validation set.


### SVM + PCA


In [10]:
pca_components = [5, 10, 15, 20, 30, 50, 80, 100, 120, 150, 200]
svm_pca_grid = [
    {"svc__kernel": ["linear"], "svc__C": [0.1, 1, 10, 15]},
    {"svc__kernel": ["rbf"], "svc__C": [1, 10], "svc__gamma": [0.001, 0.01, 0.1]},
]

best_score, best_params, best_n = -1.0, None, None
svm_pca_scores = []

for n in pca_components:
    pipe = Pipeline([
        ("pca", PCA(n_components=n, random_state=SEED)),
        ("svc", SVC()),
    ])
    search = GridSearchCV(pipe, svm_pca_grid, cv=3, scoring="accuracy", n_jobs=-1)
    search.fit(X_train_s, y_train)
    val_acc = accuracy_score(y_val, search.best_estimator_.predict(X_val_s))
    svm_pca_scores.append(val_acc)
    print(f"PCA={n:3d} | CV-best={search.best_params_} | val={val_acc:.4f}")
    if val_acc > best_score:
        best_score, best_params, best_n = val_acc, search.best_params_, n

line_accuracy_chart(
    pca_components,
    svm_pca_scores,
    "SVM + PCA — validation accuracy vs components",
    save_as="07_svm_pca_curve.png",
)

svm_pca_model = Pipeline([
    ("pca", PCA(n_components=best_n, random_state=SEED)),
    ("svc", SVC(
        kernel=best_params["svc__kernel"],
        C=best_params["svc__C"],
        gamma=best_params.get("svc__gamma", "scale"),
    )),
])
svm_pca_model.fit(X_train_s, y_train)

print(f"Selected PCA components: {best_n}")
print(f"Selected params: {best_params}")
for name, Xs, ys in [("train", X_train_s, y_train), ("validation", X_val_s, y_val)]:
    pred = svm_pca_model.predict(Xs)
    acc = accuracy_score(ys, pred)
    print(f"Accuracy ({name}): {acc:.4f}")
    if name == "validation":
        model_accuracies["SVM + PCA"] = acc
        plot_confusion(ys, pred, "SVM + PCA — validation", save_as="08_cm_svm_pca.png")


PCA=  5 | CV-best={'svc__C': 10, 'svc__gamma': 0.001, 'svc__kernel': 'rbf'} | val=0.8967


PCA= 10 | CV-best={'svc__C': 10, 'svc__kernel': 'linear'} | val=0.9133


PCA= 15 | CV-best={'svc__C': 10, 'svc__gamma': 0.001, 'svc__kernel': 'rbf'} | val=0.9200


PCA= 20 | CV-best={'svc__C': 1, 'svc__kernel': 'linear'} | val=0.9133


PCA= 30 | CV-best={'svc__C': 10, 'svc__gamma': 0.001, 'svc__kernel': 'rbf'} | val=0.9167


...

PCA=100 | CV-best={'svc__C': 10, 'svc__gamma': 0.001, 'svc__kernel': 'rbf'} | val=0.9200


PCA=120 | CV-best={'svc__C': 10, 'svc__gamma': 0.001, 'svc__kernel': 'rbf'} | val=0.9200


PCA=150 | CV-best={'svc__C': 10, 'svc__gamma': 0.001, 'svc__kernel': 'rbf'} | val=0.9233


PCA=200 | CV-best={'svc__C': 10, 'svc__gamma': 0.001, 'svc__kernel': 'rbf'} | val=0.9200


Selected PCA components: 150
Selected params: {'svc__C': 10, 'svc__gamma': 0.001, 'svc__kernel': 'rbf'}
Accuracy (train): 0.9956
Accuracy (validation): 0.9233


PCA improves the train/validation gap: training accuracy drops slightly while validation accuracy rises — discarding noisy pixel dimensions reduces overfitting.


### Naive Bayes + PCA


In [11]:
best_score, best_n = -1.0, None
nb_pca_scores = []

for n in pca_components:
    pipe = Pipeline([
        ("pca", PCA(n_components=n, random_state=SEED)),
        ("nb", GaussianNB()),
    ])
    pipe.fit(X_train_s, y_train)
    acc = accuracy_score(y_val, pipe.predict(X_val_s))
    nb_pca_scores.append(acc)
    if acc > best_score:
        best_score, best_n = acc, n

line_accuracy_chart(
    pca_components,
    nb_pca_scores,
    "Naive Bayes + PCA — validation accuracy vs components",
    save_as="09_nb_pca_curve.png",
)

nb_pca_model = Pipeline([
    ("pca", PCA(n_components=best_n, random_state=SEED)),
    ("nb", GaussianNB()),
])
nb_pca_model.fit(X_train_s, y_train)

print(f"Selected PCA components: {best_n}")
for name, Xs, ys in [("train", X_train_s, y_train), ("validation", X_val_s, y_val)]:
    pred = nb_pca_model.predict(Xs)
    acc = accuracy_score(ys, pred)
    print(f"Accuracy ({name}): {acc:.4f}")
    if name == "validation":
        model_accuracies["Naive Bayes + PCA"] = acc
        plot_confusion(ys, pred, "Naive Bayes + PCA — validation")


Selected PCA components: 20
Accuracy (train): 0.9300
Accuracy (validation): 0.8900


Unlike SVM, Naive Bayes prefers **very low** PCA dimensionality. Extra components reintroduce correlated noise that violates its independence assumption.


### LDA + PCA


In [12]:
best_score, best_params, best_n = -1.0, None, None
lda_pca_scores = []

for n in pca_components:
    best_for_n, params_for_n = -1.0, None
    for params in valid_lda_params:
        try:
            pipe = Pipeline([
                ("pca", PCA(n_components=n, random_state=SEED)),
                ("lda", LDA(**params)),
            ])
            pipe.fit(X_train_s, y_train)
            acc = accuracy_score(y_val, pipe.predict(X_val_s))
            if acc > best_for_n:
                best_for_n, params_for_n = acc, params
        except Exception:
            continue
    lda_pca_scores.append(best_for_n)
    print(f"PCA={n:3d} | params={params_for_n} | val={best_for_n:.4f}")
    if best_for_n > best_score:
        best_score, best_params, best_n = best_for_n, params_for_n, n

line_accuracy_chart(
    pca_components,
    lda_pca_scores,
    "LDA + PCA — validation accuracy vs components",
    save_as="10_lda_pca_curve.png",
)

lda_pca_model = Pipeline([
    ("pca", PCA(n_components=best_n, random_state=SEED)),
    ("lda", LDA(**best_params)),
])
lda_pca_model.fit(X_train_s, y_train)

print(f"Selected PCA components: {best_n}")
print(f"Selected params: {best_params}")
for name, Xs, ys in [("train", X_train_s, y_train), ("validation", X_val_s, y_val)]:
    pred = lda_pca_model.predict(Xs)
    acc = accuracy_score(ys, pred)
    print(f"Accuracy ({name}): {acc:.4f}")
    if name == "validation":
        model_accuracies["LDA + PCA"] = acc
        plot_confusion(ys, pred, "LDA + PCA — validation")


PCA=  5 | params={'shrinkage': None, 'solver': 'svd'} | val=0.8767


PCA= 10 | params={'shrinkage': None, 'solver': 'svd'} | val=0.9100


PCA= 15 | params={'shrinkage': None, 'solver': 'svd'} | val=0.9067


PCA= 20 | params={'shrinkage': None, 'solver': 'svd'} | val=0.9000


PCA= 30 | params={'shrinkage': None, 'solver': 'svd'} | val=0.8967


...

PCA=100 | params={'shrinkage': None, 'solver': 'svd'} | val=0.9167


PCA=120 | params={'shrinkage': None, 'solver': 'svd'} | val=0.9200


PCA=150 | params={'shrinkage': None, 'solver': 'svd'} | val=0.9133


PCA=200 | params={'shrinkage': None, 'solver': 'svd'} | val=0.9067


Selected PCA components: 80
Selected params: {'shrinkage': None, 'solver': 'svd'}
Accuracy (train): 0.9722
Accuracy (validation): 0.9200


## Dimensionality reduction with LLE

Locally Linear Embedding preserves local neighborhood geometry. We also tune the number of neighbors. LLE is slower than PCA, so the grid is focused but still explores a wide component range.


### SVM + LLE


In [13]:
lle_dims = [10, 20, 30, 50, 80, 100, 120, 150]
neighbors = [5, 10, 15, 20]
svm_lle_grid = [
    {"lle__n_neighbors": neighbors, "svc__kernel": ["linear"], "svc__C": [0.1, 1, 10, 15]},
    {
        "lle__n_neighbors": neighbors,
        "svc__kernel": ["rbf"],
        "svc__C": [1, 10],
        "svc__gamma": [0.001, 0.01, 0.1],
    },
]

best_score, best_params, best_n = -1.0, None, None
svm_lle_scores = []

for n in lle_dims:
    pipe = Pipeline([
        ("lle", LLE(n_components=n, random_state=SEED, eigen_solver="dense")),
        ("svc", SVC()),
    ])
    search = GridSearchCV(pipe, svm_lle_grid, cv=3, scoring="accuracy", n_jobs=-1)
    try:
        search.fit(X_train_s, y_train)
        val_acc = accuracy_score(y_val, search.best_estimator_.predict(X_val_s))
        print(f"LLE={n:3d} | {search.best_params_} | val={val_acc:.4f}")
    except Exception as exc:
        val_acc = 0.0
        search = None
        print(f"LLE={n:3d} failed: {exc}")
    svm_lle_scores.append(val_acc)
    if search is not None and val_acc > best_score:
        best_score, best_params, best_n = val_acc, search.best_params_, n

line_accuracy_chart(
    lle_dims,
    svm_lle_scores,
    "SVM + LLE — validation accuracy vs components",
    xlabel="LLE components",
    save_as="11_svm_lle_curve.png",
)

svm_lle_model = Pipeline([
    (
        "lle",
        LLE(
            n_components=best_n,
            n_neighbors=best_params["lle__n_neighbors"],
            random_state=SEED,
            eigen_solver="dense",
        ),
    ),
    (
        "svc",
        SVC(
            kernel=best_params["svc__kernel"],
            C=best_params["svc__C"],
            gamma=best_params.get("svc__gamma", "scale"),
        ),
    ),
])
svm_lle_model.fit(X_train_s, y_train)

print(f"Selected LLE components: {best_n}")
print(f"Selected params: {best_params}")
for name, Xs, ys in [("train", X_train_s, y_train), ("validation", X_val_s, y_val)]:
    pred = svm_lle_model.predict(Xs)
    acc = accuracy_score(ys, pred)
    print(f"Accuracy ({name}): {acc:.4f}")
    if name == "validation":
        model_accuracies["SVM + LLE"] = acc
        plot_confusion(ys, pred, "SVM + LLE — validation")


LLE= 10 | {'lle__n_neighbors': 10, 'svc__C': 15, 'svc__kernel': 'linear'} | val=0.8767


LLE= 20 | {'lle__n_neighbors': 15, 'svc__C': 15, 'svc__kernel': 'linear'} | val=0.9200


LLE= 30 | {'lle__n_neighbors': 20, 'svc__C': 15, 'svc__kernel': 'linear'} | val=0.9233


LLE= 50 | {'lle__n_neighbors': 10, 'svc__C': 10, 'svc__kernel': 'linear'} | val=0.9000


LLE= 80 | {'lle__n_neighbors': 20, 'svc__C': 15, 'svc__kernel': 'linear'} | val=0.9067


LLE=100 | {'lle__n_neighbors': 20, 'svc__C': 15, 'svc__kernel': 'linear'} | val=0.8967


LLE=120 | {'lle__n_neighbors': 20, 'svc__C': 1, 'svc__kernel': 'linear'} | val=0.8933


LLE=150 | {'lle__n_neighbors': 20, 'svc__C': 1, 'svc__kernel': 'linear'} | val=0.8967


Selected LLE components: 30
Selected params: {'lle__n_neighbors': 20, 'svc__C': 15, 'svc__kernel': 'linear'}
Accuracy (train): 0.9544
Accuracy (validation): 0.9233


### Naive Bayes + LLE


In [14]:
best_score, best_params = -1.0, None
nb_lle_scores = []

for n in lle_dims:
    best_for_n, k_for_n = -1.0, None
    for k in neighbors:
        if k >= n:
            continue
        try:
            pipe = Pipeline([
                (
                    "lle",
                    LLE(
                        n_components=n,
                        n_neighbors=k,
                        random_state=SEED,
                        eigen_solver="dense",
                    ),
                ),
                ("nb", GaussianNB()),
            ])
            pipe.fit(X_train_s, y_train)
            acc = accuracy_score(y_val, pipe.predict(X_val_s))
            if acc > best_for_n:
                best_for_n, k_for_n = acc, k
        except Exception:
            continue
    nb_lle_scores.append(best_for_n if best_for_n >= 0 else np.nan)
    print(f"LLE={n:3d} | neighbors={k_for_n} | val={best_for_n:.4f}")
    if best_for_n > best_score:
        best_score = best_for_n
        best_params = {"n_components": n, "n_neighbors": k_for_n}

line_accuracy_chart(
    lle_dims,
    nb_lle_scores,
    "Naive Bayes + LLE — validation accuracy vs components",
    xlabel="LLE components",
    save_as="12_nb_lle_curve.png",
)

nb_lle_model = Pipeline([
    (
        "lle",
        LLE(
            n_components=best_params["n_components"],
            n_neighbors=best_params["n_neighbors"],
            random_state=SEED,
            eigen_solver="dense",
        ),
    ),
    ("nb", GaussianNB()),
])
nb_lle_model.fit(X_train_s, y_train)

print("Selected:", best_params)
for name, Xs, ys in [("train", X_train_s, y_train), ("validation", X_val_s, y_val)]:
    pred = nb_lle_model.predict(Xs)
    acc = accuracy_score(ys, pred)
    print(f"Accuracy ({name}): {acc:.4f}")
    if name == "validation":
        model_accuracies["Naive Bayes + LLE"] = acc
        plot_confusion(ys, pred, "Naive Bayes + LLE — validation")


LLE= 10 | neighbors=5 | val=0.8633


LLE= 20 | neighbors=15 | val=0.8800


LLE= 30 | neighbors=20 | val=0.9000


LLE= 50 | neighbors=20 | val=0.9000


LLE= 80 | neighbors=20 | val=0.9000


LLE=100 | neighbors=5 | val=0.9000


LLE=120 | neighbors=20 | val=0.9000


LLE=150 | neighbors=20 | val=0.9067


Selected: {'n_components': 150, 'n_neighbors': 20}
Accuracy (train): 0.9133
Accuracy (validation): 0.9067


With LLE, Naive Bayes often prefers **higher** dimensionality than with PCA, but overall accuracy still trails the PCA variant.


### LDA + LLE


In [15]:
best_score, best_params, best_n = -1.0, None, None
lda_lle_scores = []

for n in lle_dims:
    best_for_n, params_for_n = -1.0, None
    for k in neighbors:
        if k >= n:
            continue
        for lda_params in valid_lda_params:
            try:
                pipe = Pipeline([
                    (
                        "lle",
                        LLE(
                            n_components=n,
                            n_neighbors=k,
                            random_state=SEED,
                            eigen_solver="dense",
                        ),
                    ),
                    ("lda", LDA(**lda_params)),
                ])
                pipe.fit(X_train_s, y_train)
                acc = accuracy_score(y_val, pipe.predict(X_val_s))
                if acc > best_for_n:
                    best_for_n = acc
                    params_for_n = {
                        "lle__n_neighbors": k,
                        **{f"lda__{kk}": vv for kk, vv in lda_params.items()},
                    }
            except Exception:
                continue
    lda_lle_scores.append(best_for_n if best_for_n >= 0 else np.nan)
    print(f"LLE={n:3d} | {params_for_n} | val={best_for_n:.4f}")
    if best_for_n > best_score:
        best_score, best_params, best_n = best_for_n, params_for_n, n

line_accuracy_chart(
    lle_dims,
    lda_lle_scores,
    "LDA + LLE — validation accuracy vs components",
    xlabel="LLE components",
    save_as="13_lda_lle_curve.png",
)

lda_lle_model = Pipeline([
    (
        "lle",
        LLE(
            n_components=best_n,
            n_neighbors=best_params["lle__n_neighbors"],
            random_state=SEED,
            eigen_solver="dense",
        ),
    ),
    (
        "lda",
        LDA(
            solver=best_params["lda__solver"],
            shrinkage=best_params["lda__shrinkage"],
        ),
    ),
])
lda_lle_model.fit(X_train_s, y_train)

print(f"Selected LLE components: {best_n}")
print(f"Selected params: {best_params}")
for name, Xs, ys in [("train", X_train_s, y_train), ("validation", X_val_s, y_val)]:
    pred = lda_lle_model.predict(Xs)
    acc = accuracy_score(ys, pred)
    print(f"Accuracy ({name}): {acc:.4f}")
    if name == "validation":
        model_accuracies["LDA + LLE"] = acc
        plot_confusion(ys, pred, "LDA + LLE — validation")


LLE= 10 | {'lle__n_neighbors': 5, 'lda__shrinkage': 'auto', 'lda__solver': 'lsqr'} | val=0.8767


LLE= 20 | {'lle__n_neighbors': 15, 'lda__shrinkage': None, 'lda__solver': 'svd'} | val=0.8933


LLE= 30 | {'lle__n_neighbors': 20, 'lda__shrinkage': None, 'lda__solver': 'svd'} | val=0.9133


LLE= 50 | {'lle__n_neighbors': 15, 'lda__shrinkage': 'auto', 'lda__solver': 'lsqr'} | val=0.9067


LLE= 80 | {'lle__n_neighbors': 15, 'lda__shrinkage': None, 'lda__solver': 'svd'} | val=0.9133


LLE=100 | {'lle__n_neighbors': 15, 'lda__shrinkage': 'auto', 'lda__solver': 'lsqr'} | val=0.9167


LLE=120 | {'lle__n_neighbors': 5, 'lda__shrinkage': None, 'lda__solver': 'svd'} | val=0.9067


LLE=150 | {'lle__n_neighbors': 10, 'lda__shrinkage': None, 'lda__solver': 'svd'} | val=0.9067


Selected LLE components: 100
Selected params: {'lle__n_neighbors': 15, 'lda__shrinkage': 'auto', 'lda__solver': 'lsqr'}
Accuracy (train): 0.9678
Accuracy (validation): 0.9167


## Model comparison and final evaluation

We select the best pipeline by **validation accuracy**, then estimate real-world performance once on the untouched test set.


In [16]:
comparison = pd.Series(model_accuracies, name="validation_accuracy").sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 4.5))
colors = [
    PALETTE["med_blue"] if m == comparison.index[-1] else PALETTE["slate"]
    for m in comparison.index
]
ax.barh(comparison.index, comparison.values, color=colors, height=0.65)
ax.set_xlabel("Validation accuracy")
ax.set_title("Model comparison")
ax.set_xlim(0.8, 1.0)
for i, (name, val) in enumerate(comparison.items()):
    ax.text(val + 0.002, i, f"{val:.3f}", va="center", fontsize=9, color=PALETTE["slate"])
fig.tight_layout()
fig.savefig(FIGURES_DIR / "14_model_comparison.png", bbox_inches="tight")
plt.show()

best_name = comparison.index[-1]
print(f"Best validation model: {best_name} ({comparison.iloc[-1]:.4f})")

final_model = {
    "SVM": svm_model,
    "Naive Bayes": nb_model,
    "LDA": lda_model,
    "SVM + PCA": svm_pca_model,
    "Naive Bayes + PCA": nb_pca_model,
    "LDA + PCA": lda_pca_model,
    "SVM + LLE": svm_lle_model,
    "Naive Bayes + LLE": nb_lle_model,
    "LDA + LLE": lda_lle_model,
}[best_name]

uses_scaled = best_name != "Naive Bayes"
X_test_final = X_test_s if uses_scaled else X_test

y_test_pred = final_model.predict(X_test_final)
test_acc = accuracy_score(y_test, y_test_pred)
print(f"Held-out test accuracy: {test_acc:.4f}")
plot_confusion(
    y_test,
    y_test_pred,
    f"{best_name} — test set",
    save_as="15_cm_final_test.png",
)



Best validation model: SVM + LLE (0.9233)
Held-out test accuracy: 0.9433


array([[139,  13],
       [  4, 144]])

## Takeaways

1. **Dimensionality reduction helps.** Full-pixel SVM overfits; PCA recovers validation accuracy.
2. **SVM + PCA is typically the strongest pipeline** among the nine combinations tried here.
3. **Naive Bayes needs aggressive compression** (or better features) — raw pixels violate independence.
4. **LLE is competitive but slower**, and did not consistently beat PCA for SVM on this dataset.
5. Expected performance on new data is around **93% accuracy**, based on the held-out test set.

Possible next steps: CNN features, data augmentation, or probability calibration for threshold tuning.
